# Lab 28 - Colab GPU Setup Option A

Single GPU vLLM setup for Google Colab. This notebook exposes vLLM and the embedding service through ngrok so your local Lab 28 stack can call them through `.env`.

## 0. Runtime Setup

In Colab, go to **Runtime -> Change runtime type -> GPU** before running. A T4 GPU is enough for this fallback setup.

In [5]:
# 1. Install stable fallback dependencies for Colab/T4
# After this cell finishes, restart the Colab runtime before importing vLLM.

%pip uninstall -y vllm flashinfer-python transformers tokenizers
%pip install -q --force-reinstall --no-cache-dir "numpy==1.26.4"
%pip install -q "transformers==4.48.3" "tokenizers==0.21.1"
%pip install -q "vllm==0.7.3" fastapi uvicorn pyngrok mlflow sentence-transformers requests

print("Install finished.")
print("Now click Runtime -> Restart runtime, then continue from the next cell.")

Found existing installation: vllm 0.7.3
Uninstalling vllm-0.7.3:
  Successfully uninstalled vllm-0.7.3
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 262.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires tokenizers, which is not installed.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, which is not installed.
peft 0.19.1 requires transformers, which is not installed.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requir

## Restart Required

Stop here after the install cell. Restart the runtime, then run the remaining cells. This avoids NumPy/PyTorch binary mismatch errors after pip changes core packages.

In [2]:
# 2. Verify GPU and package versions after restart
import numpy as np
import torch
import vllm

print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("vLLM:", vllm.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Enable GPU runtime before continuing."

numpy: 1.26.4
torch: 2.5.1+cu124
vLLM: 0.7.3
cuda available: True
gpu: Tesla T4


In [3]:
# 3. Load ngrok token from Colab Secrets
# In Colab, open the key icon on the left and add a secret named NGROK_TOKEN.
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_TOKEN")
except Exception:
    NGROK_TOKEN = None

# Fallback: paste your token here if you do not want to use Colab Secrets.
if not NGROK_TOKEN:
    NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

assert NGROK_TOKEN != "PASTE_YOUR_NGROK_TOKEN_HERE", "Set NGROK_TOKEN in Colab Secrets or paste it above."

In [15]:
# 4. Configuration
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"
VLLM_PORT = 8001
EMBED_PORT = 8003

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print("ngrok token configured")

ngrok token configured


In [5]:
# 5. Start vLLM server on one GPU
import os
import socket
import subprocess
import threading
import time
import requests

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["VLLM_USE_V1"] = "0"

vllm_process = None

def stream_process(prefix, process):
    for line in iter(process.stdout.readline, b""):
        print(f"[{prefix}] {line.decode(errors='replace')}", end="")

def port_is_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex(("127.0.0.1", port)) == 0

def stop_existing_vllm():
    global vllm_process
    if vllm_process and vllm_process.poll() is None:
        print("Stopping previous vLLM process...")
        vllm_process.terminate()
        try:
            vllm_process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            vllm_process.kill()

def start_vllm():
    global vllm_process
    stop_existing_vllm()
    cmd = [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_NAME,
        "--host", "0.0.0.0",
        "--port", str(VLLM_PORT),
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.6",
        "--trust-remote-code",
    ]
    print("Starting:", " ".join(cmd))
    vllm_process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
        bufsize=1,
    )
    threading.Thread(target=stream_process, args=("vLLM", vllm_process), daemon=True).start()
    return vllm_process

def wait_for_vllm(timeout_seconds=1200):
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        if vllm_process and vllm_process.poll() is not None:
            raise RuntimeError(f"vLLM process exited with code {vllm_process.returncode}. Check the [vLLM] logs above.")

        for path in ("/health", "/v1/models"):
            try:
                response = requests.get(f"http://127.0.0.1:{VLLM_PORT}{path}", timeout=5)
                if response.status_code == 200:
                    print(f"vLLM server is ready via {path}")
                    return
            except requests.RequestException:
                pass

        print(f"Waiting for vLLM to load... port_open={port_is_open(VLLM_PORT)}")
        time.sleep(10)

    raise RuntimeError("vLLM did not become ready in time. Check the [vLLM] logs above.")

start_vllm()
wait_for_vllm()

Starting: python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4 --host 0.0.0.0 --port 8001 --max-model-len 2048 --gpu-memory-utilization 0.6 --trust-remote-code
Waiting for vLLM to load... port_open=False


/usr/lib/python3.12/subprocess.py:1016: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stdout = io.open(c2pread, 'rb', bufsize)


Waiting for vLLM to load... port_open=False
[vLLM] INFO 05-18 16:30:18 __init__.py:207] Automatically detected platform cuda.
[vLLM] INFO 05-18 16:30:18 api_server.py:912] vLLM API server version 0.7.3
[vLLM] INFO 05-18 16:30:18 api_server.py:913] args: Namespace(host='0.0.0.0', port=8001, uvicorn_log_level='info', allow_credentials=False, allowed_origins=['*'], allowed_methods=['*'], allowed_headers=['*'], api_key=None, lora_modules=None, prompt_adapters=None, chat_template=None, chat_template_content_format='auto', response_role='assistant', ssl_keyfile=None, ssl_certfile=None, ssl_ca_certs=None, ssl_cert_reqs=0, root_path=None, middleware=[], return_tokens_as_token_ids=False, disable_frontend_multiprocessing=False, enable_request_id_headers=False, enable_auto_tool_choice=False, enable_reasoning=False, reasoning_parser=None, tool_call_parser=None, tool_parser_plugin='', model='Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4', task='auto', tokenizer=None, skip_tokenizer_init=False, revision=None, 

In [6]:
# 6. Expose vLLM with ngrok
vllm_tunnel = ngrok.connect(VLLM_PORT, "http")
VLLM_NGROK_URL = vllm_tunnel.public_url

print("Copy this into local .env:")print(f"VLLM_NGROK_URL={VLLM_NGROK_URL}")

Copy this into local .env:
VLLM_NGROK_URL=https://nutcase-monsieur-cilantro.ngrok-free.dev


In [7]:
# 7. Quick vLLM smoke test
payload = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Reply with one short sentence about platform engineering."}],
}

response = requests.post(f"{VLLM_NGROK_URL}/v1/chat/completions", json=payload, timeout=120)
print("status:", response.status_code)
print(response.text[:500])
response.raise_for_status()print(response.json()["choices"][0]["message"]["content"])

[vLLM] INFO 05-18 16:37:24 chat_utils.py:332] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
[vLLM] INFO 05-18 16:37:24 logger.py:39] Received request chatcmpl-ee781b41434c4a5caf6c9eb2f2d3e14c: prompt: '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nReply with one short sentence about platform engineering.<|im_end|>\n<|im_start|>assistant\n', params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=1.0, top_p=1.0, top_k=-1, min_p=0.0, seed=None, stop=[], stop_token_ids=[], bad_words=[], include_stop_str_in_output=False, ignore_eos=False, max_tokens=2010, min_tokens=0, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, truncate_prompt_tokens=None, guided_decoding=None), prompt_token_ids: None, lora_request: None, prompt_adapter_request: None.
[vL

In [13]:
import requests
for url in ["http://127.0.0.1:8001/health", "http://127.0.0.1:8001/v1/models"]:
    try:
        r = requests.get(url, timeout=10)
        print(url, r.status_code, r.text[:300])
    except Exception as e:
        print(url, e)

[vLLM] INFO:     127.0.0.1:58648 - "GET /health HTTP/1.1" 200 OK
http://127.0.0.1:8001/health 200 
[vLLM] INFO:     127.0.0.1:58650 - "GET /v1/models HTTP/1.1" 200 OK
http://127.0.0.1:8001/v1/models 200 {"object":"list","data":[{"id":"Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4","object":"model","created":1779123269,"owned_by":"vllm","root":"Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4","parent":null,"max_model_len":2048,"permission":[{"id":"modelperm-371d235a17af491cbcd9a8e28e5547d4","object":"model_permission","cre


## Embedding Service

Run these cells too if you want `EMBED_NGROK_URL` for `scripts/05_embed_to_qdrant.py`.

In [16]:
# 8. Start combined proxy API server
# This exposes both OpenAI-compatible vLLM routes and /embed through one ngrok URL.
# Use the printed proxy URL for both VLLM_NGROK_URL and EMBED_NGROK_URL.
from fastapi import FastAPI, Request
import hashlib
import math
import requests
import socket
import uvicorn

PROXY_PORT = EMBED_PORT
VECTOR_SIZE = 384
proxy_app = FastAPI(title="Lab28 Colab Proxy")

def deterministic_embedding(text: str, size: int = VECTOR_SIZE):
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    values = []
    while len(values) < size:
        for byte in digest:
            values.append((byte / 255.0) * 2 - 1)
            if len(values) == size:
                break
        digest = hashlib.sha256(digest).digest()
    norm = math.sqrt(sum(value * value for value in values)) or 1.0
    return [value / norm for value in values]

def wait_for_port(port, timeout_seconds=30):
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(1)
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(1)
    return False

@proxy_app.get("/health")
def proxy_health():
    upstream = requests.get(f"http://127.0.0.1:{VLLM_PORT}/health", timeout=10)
    upstream.raise_for_status()
    return {"status": "ok", "upstream": "vllm"}

@proxy_app.get("/v1/models")
def proxy_models():
    response = requests.get(f"http://127.0.0.1:{VLLM_PORT}/v1/models", timeout=30)
    response.raise_for_status()
    return response.json()

@proxy_app.post("/v1/chat/completions")
async def proxy_chat(request: Request):
    body = await request.json()
    response = requests.post(f"http://127.0.0.1:{VLLM_PORT}/v1/chat/completions", json=body, timeout=180)
    response.raise_for_status()
    return response.json()

@proxy_app.post("/embed")
def embed(data: dict):
    texts = data["texts"]
    try:
        response = requests.post(
            f"http://127.0.0.1:{VLLM_PORT}/v1/embeddings",
            json={"model": MODEL_NAME, "input": texts},
            timeout=60,
        )
        if response.status_code == 200:
            payload = response.json()
            embeddings = [item["embedding"] for item in payload.get("data", [])]
            if len(embeddings) == len(texts):
                return {"embeddings": embeddings, "source": "vllm"}
    except Exception as exc:
        print("vLLM embedding fallback:", exc)
    return {"embeddings": [deterministic_embedding(text) for text in texts], "source": "deterministic"}

def run_proxy_server():
    uvicorn.run(proxy_app, host="0.0.0.0", port=PROXY_PORT)

threading.Thread(target=run_proxy_server, daemon=True).start()
assert wait_for_port(PROXY_PORT), f"Proxy did not open port {PROXY_PORT}"

health = requests.get(f"http://127.0.0.1:{PROXY_PORT}/health", timeout=30)
health.raise_for_status()
embed_health = requests.post(f"http://127.0.0.1:{PROXY_PORT}/embed", json={"texts": ["health check"]}, timeout=30)
embed_health.raise_for_status()
print("Combined proxy started")
print("proxy port:", PROXY_PORT)
print("embedding source:", embed_health.json().get("source"))
print("embedding dimensions:", len(embed_health.json()["embeddings"][0]))

INFO:     Started server process [14091]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8003 (Press CTRL+C to quit)


[vLLM] INFO:     127.0.0.1:57848 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:57626 - "GET /health HTTP/1.1" 200 OK
[vLLM] INFO:     127.0.0.1:57864 - "POST /v1/embeddings HTTP/1.1" 200 OK
INFO:     127.0.0.1:57634 - "POST /embed HTTP/1.1" 200 OK
Combined proxy started
proxy port: 8003
embedding source: deterministic
embedding dimensions: 384


In [17]:
# 9. Expose embedding service with ngrok
embed_tunnel = ngrok.connect(EMBED_PORT, "http")
EMBED_NGROK_URL = embed_tunnel.public_url

print("Copy these into local .env:")
print(f"VLLM_NGROK_URL={VLLM_NGROK_URL}")
print(f"EMBED_NGROK_URL={EMBED_NGROK_URL}")
print(f"MODEL_NAME={MODEL_NAME}")

Copy these into local .env:
VLLM_NGROK_URL=https://nutcase-monsieur-cilantro.ngrok-free.dev
EMBED_NGROK_URL=https://nutcase-monsieur-cilantro.ngrok-free.dev
MODEL_NAME=Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4


In [18]:
# 10. Quick embedding smoke test
response = requests.post(f"{EMBED_NGROK_URL}/embed", json={"texts": ["AI platform integration test"]}, timeout=30)
print("status:", response.status_code)
response.raise_for_status()
embedding = response.json()["embeddings"][0]
print("embedding dimensions:", len(embedding))

[vLLM] INFO:     127.0.0.1:48176 - "POST /v1/embeddings HTTP/1.1" 200 OK
INFO:     34.16.154.99:0 - "POST /embed HTTP/1.1" 200 OK
status: 200
embedding dimensions: 384
